# QUANT Classifier on 5 Standardized SWAN-SF Features — TSS-Optimized Version

This notebook trains an **aeon `QUANTClassifier`** using only these five standardized SHARP/SWAN-SF features:

1. `TOTUSJH` — Total unsigned current helicity
2. `TOTBSQ` — Total magnitude of Lorentz force
3. `TOTPOT` — Total photospheric magnetic free energy density
4. `TOTUSJZ` — Total unsigned vertical current
5. `ABSNJZH` — Absolute value of the net current helicity

This version is designed for the second experiment: **optimize the decision threshold for TSS**.

Main workflow:

```text
training data -> stratified fit/validation split
fit QUANT on fit subset
use validation probabilities to choose the threshold that maximizes TSS
apply that threshold once to the held-out test set
```

Important: the threshold is chosen from the validation split only, not from the test set. This avoids leaking test-set information into model selection.

Expected input files:

- `X_train.npy`
- `X_test.npy`
- `y_train.npy`
- `y_test.npy`
- `feature_columns_final_magnetic.json`

The expected aeon tensor shape is `(n_cases, n_channels, n_timepoints)`, for example `(277010, 47, 60)`. The notebook also handles the alternate shape `(n_cases, n_timepoints, n_channels)` by transposing after feature selection.


## 1. Install and import packages

Run this cell first. It installs `aeon` only if it is not already available in the environment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import importlib.util
import subprocess
import sys

# Install aeon only when needed. This keeps the notebook usable in fresh Colab sessions.
if importlib.util.find_spec("aeon") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "aeon", "scikit-learn", "joblib", "pandas"])
else:
    print("aeon is already installed.")


In [ ]:
from pathlib import Path
import gc
import json
import time

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.ensemble import ExtraTreesClassifier
from aeon.classification.interval_based import QUANTClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

print("Imports complete.")


## 2. Configure paths, features, sampling, and TSS tuning

Edit `DATA_DIR` so it points to the folder containing your standardized tensors and label arrays.

The settings below let you run either a quick sample experiment or the full dataset. For the full dataset, set both sample sizes to `None`.


In [ ]:
# =============================
# Main path configuration
# =============================
# Change this to the folder where you saved the standardized dataset.
# Example for Google Drive:
DATA_DIR = Path("/content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only")

OUTPUT_DIR = DATA_DIR / "quant_5_feature_tss_optimized_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The five channels requested for this QUANT model.
FEATURES_TO_USE = [
    "TOTUSJH",
    "TOTBSQ",
    "TOTPOT",
    "TOTUSJZ",
    "ABSNJZH",
]

# Optional quick test settings.
# Set TRAIN_SAMPLE_SIZE = None and TEST_SAMPLE_SIZE = None to use the full dataset.
TRAIN_SAMPLE_SIZE = None    # Example: 20_000 or 50_000 for a quick training test; None for full train set
TEST_SAMPLE_SIZE = None     # Example: 5_000 or 10_000 for quick evaluation; None for full test set
RANDOM_SEED = 42

# Validation split used only for threshold selection.
# The test set is not used to choose the threshold.
VALIDATION_SIZE = 0.20
OPTIMIZE_THRESHOLD_FOR = "TSS"
DEFAULT_PROBA_THRESHOLD = 0.50
PREDICT_BATCH_SIZE = 20000

# Optional second fit after choosing the threshold.
# Recommended starting point: False, because it keeps probability calibration tied to the validation-tuned model.
# If True, the notebook refits QUANT on all sampled training data after threshold selection, then applies the same threshold to test.
REFIT_ON_FULL_TRAIN_AFTER_THRESHOLD = True

# QUANT / aeon cannot handle missing values or near-constant case/channel pairs.
# This checks and removes invalid cases before fit/predict so the notebook runs cleanly.
DROP_LOW_VARIANCE_CASES = True
VARIANCE_THRESHOLD = 1e-7

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Features: {FEATURES_TO_USE}")
print(f"Validation size: {VALIDATION_SIZE}")
print(f"Threshold objective: {OPTIMIZE_THRESHOLD_FOR}")


## 3. Locate and load the standardized dataset

This cell tries the filenames used in your standardized dataset notebook. Adjust the candidate lists only if your files have different names.


In [ ]:
def find_first_existing(data_dir: Path, candidates: list[str]) -> Path:
    """Return the first existing file path from a list of candidate filenames."""
    for name in candidates:
        path = data_dir / name
        if path.exists():
            return path
    candidate_text = "\n".join(str(data_dir / name) for name in candidates)
    raise FileNotFoundError(
        "None of the expected files were found. Checked:\n" + candidate_text
    )

x_train_path = find_first_existing(DATA_DIR, [
    "X_train.npy",
    "X_train_standardized_aeon.npy",
])
x_test_path = find_first_existing(DATA_DIR, [
    "X_test.npy",
    "X_test_standardized_aeon.npy",
])
y_train_path = find_first_existing(DATA_DIR, [
    "y_train.npy",
])
y_test_path = find_first_existing(DATA_DIR, [
    "y_test.npy",
])
feature_cols_path = find_first_existing(DATA_DIR, [
    "feature_columns_final_magnetic.json",
    "feature_columns.json",
])

print("Found files:")
print("X train:", x_train_path)
print("X test: ", x_test_path)
print("y train:", y_train_path)
print("y test: ", y_test_path)
print("features:", feature_cols_path)


In [ ]:
# Memory-map the large X arrays first. The selected 5 channels are copied into memory later.
X_train_raw = np.load(x_train_path, mmap_mode="r")
X_test_raw = np.load(x_test_path, mmap_mode="r")
y_train = np.load(y_train_path)
y_test = np.load(y_test_path)

with open(feature_cols_path, "r") as f:
    feature_columns = json.load(f)

print("Raw tensor shapes:")
print("X_train_raw:", X_train_raw.shape, X_train_raw.dtype)
print("X_test_raw: ", X_test_raw.shape, X_test_raw.dtype)
print("y_train:    ", y_train.shape, y_train.dtype)
print("y_test:     ", y_test.shape, y_test.dtype)
print("Number of feature columns:", len(feature_columns))
print("First 10 feature columns:", feature_columns[:10])


## 4. Select the five requested channels

The aeon input shape should be:

```text
(n_cases, n_channels, n_timepoints)
```

After this step, the selected tensors should have shape:

```text
(n_cases, 5, 60)
```

assuming your standardized windows are 60 time steps long.


In [ ]:
missing_features = [feat for feat in FEATURES_TO_USE if feat not in feature_columns]
if missing_features:
    raise ValueError(f"These requested features are missing from feature_columns.json: {missing_features}")

print("Mapping selected feature indices...")
selected_indices = []
feature_index_map = {}
for feat in tqdm(FEATURES_TO_USE, desc="Features"):
    idx = feature_columns.index(feat)
    selected_indices.append(idx)
    feature_index_map[feat] = idx

print("Selected feature indices:")
for feat, idx in feature_index_map.items():
    print(f"  {feat:8s} -> channel index {idx}")

In [ ]:
def select_channels_as_aeon(X, selected_indices, n_total_features):
    """Select requested channels and return shape (n_cases, n_selected_channels, n_timepoints)."""
    if X.ndim != 3:
        raise ValueError(f"Expected a 3D tensor, but got shape {X.shape}")

    # Preferred aeon layout: (n_cases, n_channels, n_timepoints)
    if X.shape[1] == n_total_features:
        X_selected = X[:, selected_indices, :]
        layout = "channels_second / aeon format"

    # Alternate layout: (n_cases, n_timepoints, n_channels)
    elif X.shape[2] == n_total_features:
        X_selected = X[:, :, selected_indices].transpose(0, 2, 1)
        layout = "channels_last -> transposed to aeon format"

    else:
        raise ValueError(
            f"Could not identify channel axis. X shape is {X.shape}, "
            f"but feature_columns has length {n_total_features}."
        )

    # Convert to float32 to keep memory lower and match the saved standardized tensors.
    return np.asarray(X_selected, dtype=np.float32), layout

X_train_5, train_layout = select_channels_as_aeon(X_train_raw, selected_indices, len(feature_columns))
X_test_5, test_layout = select_channels_as_aeon(X_test_raw, selected_indices, len(feature_columns))

print("Train layout:", train_layout)
print("Test layout: ", test_layout)
print("X_train_5:", X_train_5.shape, X_train_5.dtype)
print("X_test_5: ", X_test_5.shape, X_test_5.dtype)

# Free raw memory-map handles if they are no longer needed.
del X_train_raw, X_test_raw
gc.collect()


## 5. Basic dataset checks

This verifies:

- `X` and `y` have matching case counts
- the tensors contain finite values
- class counts are visible before training


In [ ]:
def print_class_counts(y, name):
    values, counts = np.unique(y, return_counts=True)
    df = pd.DataFrame({"class": values, "count": counts})
    df["percent"] = 100 * df["count"] / len(y)
    print(f"\n{name} class counts:")
    display(df)

if X_train_5.shape[0] != len(y_train):
    raise ValueError(f"Train case mismatch: X has {X_train_5.shape[0]} cases, y has {len(y_train)}")
if X_test_5.shape[0] != len(y_test):
    raise ValueError(f"Test case mismatch: X has {X_test_5.shape[0]} cases, y has {len(y_test)}")

if not np.isfinite(X_train_5).all():
    raise ValueError("X_train_5 contains NaN or infinite values. QUANT cannot handle missing values.")
if not np.isfinite(X_test_5).all():
    raise ValueError("X_test_5 contains NaN or infinite values. QUANT cannot handle missing values.")

print("Shape checks passed.")
print_class_counts(y_train, "Train")
print_class_counts(y_test, "Test")


## 6. Optional stratified sampling

Leave `TRAIN_SAMPLE_SIZE = None` and `TEST_SAMPLE_SIZE = None` to use the full dataset.

Sampling is useful only for a quick smoke test before running QUANT on the full standardized dataset.


In [ ]:
def stratified_sample(X, y, sample_size, random_seed=42):
    """Return a stratified subset of X and y. If sample_size is None, return all cases."""
    if sample_size is None or sample_size >= len(y):
        return X, y, np.arange(len(y))

    indices = np.arange(len(y))
    sample_idx, _ = train_test_split(
        indices,
        train_size=sample_size,
        stratify=y,
        random_state=random_seed,
    )
    sample_idx = np.sort(sample_idx)
    return X[sample_idx], y[sample_idx], sample_idx

X_train_fit, y_train_fit, train_used_idx = stratified_sample(
    X_train_5, y_train, TRAIN_SAMPLE_SIZE, RANDOM_SEED
)
X_test_eval, y_test_eval, test_used_idx = stratified_sample(
    X_test_5, y_test, TEST_SAMPLE_SIZE, RANDOM_SEED
)

print("Training set used:", X_train_fit.shape, y_train_fit.shape)
print("Test set used:    ", X_test_eval.shape, y_test_eval.shape)
print_class_counts(y_train_fit, "Train used")
print_class_counts(y_test_eval, "Test used")


## 7. Check for low-variance case/channel pairs

aeon's collection validation can raise an error when any individual case/channel has too little variation. This is separate from model hyperparameters; it is an input-data validity check.

By default, this notebook removes affected cases from train and test before calling QUANT. The removed counts are printed so the change is transparent.


In [ ]:
def remove_low_variance_cases(X, y, threshold=1e-7, name="dataset"):
    """Remove cases where any selected channel has std <= threshold across time."""
    # Since np.std over the whole axis is fast, we wrap the logic in a simple progress-monitored step
    print(f"Calculating variance for {name}...")
    per_case_channel_std = np.std(X, axis=2)

    keep_mask = []
    for row in tqdm(per_case_channel_std, desc=f"Checking {name} cases"):
        keep_mask.append((row > threshold).all())

    keep_mask = np.array(keep_mask)
    n_removed = int((~keep_mask).sum())

    print(f"{name}: {n_removed} / {len(y)} cases have at least one low-variance selected channel.")

    if n_removed > 0:
        bad_pairs = np.argwhere(per_case_channel_std <= threshold)
        preview = bad_pairs[:10]
        print("First low-variance case/channel pairs shown as [case_index, selected_channel_index]:")
        print(preview)
        print("Selected channel order:", FEATURES_TO_USE)

    return X[keep_mask], y[keep_mask], keep_mask

if DROP_LOW_VARIANCE_CASES:
    X_train_fit, y_train_fit, train_keep_mask = remove_low_variance_cases(
        X_train_fit, y_train_fit, VARIANCE_THRESHOLD, "train"
    )
    X_test_eval, y_test_eval, test_keep_mask = remove_low_variance_cases(
        X_test_eval, y_test_eval, VARIANCE_THRESHOLD, "test"
    )
else:
    print("Low-variance cases were not removed because DROP_LOW_VARIANCE_CASES = False.")

print("Final training shape:", X_train_fit.shape, y_train_fit.shape)
print("Final test shape:    ", X_test_eval.shape, y_test_eval.shape)
print_class_counts(y_train_fit, "Final train")
print_class_counts(y_test_eval, "Final test")

## 8. Create a stratified validation split for TSS threshold tuning

The model is fitted on `X_model_train` and the threshold is chosen using `X_val`.

The held-out test set remains untouched until final evaluation.


In [ ]:
# Determine the positive label for the binary flare task.
all_labels_seen = np.unique(np.concatenate([y_train_fit, y_test_eval]))
positive_label = 1 if 1 in all_labels_seen else all_labels_seen[-1]
negative_label_candidates = [label for label in all_labels_seen if label != positive_label]
if len(negative_label_candidates) != 1:
    raise ValueError(f"Expected binary labels, but found labels: {all_labels_seen}")
negative_label = negative_label_candidates[0]

print("Using positive label:", positive_label)
print("Using negative label:", negative_label)

X_model_train, X_val, y_model_train, y_val = train_test_split(
    X_train_fit,
    y_train_fit,
    test_size=VALIDATION_SIZE,
    stratify=y_train_fit,
    random_state=RANDOM_SEED,
)

print("Model train shape:", X_model_train.shape, y_model_train.shape)
print("Validation shape: ", X_val.shape, y_val.shape)
print_class_counts(y_model_train, "Model train split")
print_class_counts(y_val, "Validation split")


## 9. Train QUANT on the model-training split

This notebook keeps the **QUANT interval settings at their aeon defaults** and uses an `ExtraTreesClassifier` with 200 trees and `n_jobs=-1` so the tree training can use available CPU cores.

This is the same model family as the default QUANT estimator, but with explicit parallel CPU usage.


In [ ]:
quant = QUANTClassifier(
    estimator=ExtraTreesClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    ),
    random_state=RANDOM_SEED,
)

print("QUANT parameters:")
print(quant.get_params())

start_time = time.time()
quant.fit(X_model_train, y_model_train)
fit_seconds = time.time() - start_time

print(f"QUANT fit time: {fit_seconds / 60:.2f} minutes ({fit_seconds:.1f} seconds)")


## 10. Helper functions for probabilities, thresholds, and flare skill scores

The key optimization is based on validation-set probabilities.

For a threshold `t`, the positive prediction rule is:

```python
y_pred = 1 if P(class 1) >= t else 0
```

TSS is computed as:

```text
TSS = POD - FPR
```


In [ ]:
def predict_proba_in_batches(model, X, batch_size=20000):
    """Predict class probabilities in batches to reduce memory spikes."""
    chunks = []
    for start in tqdm(range(0, X.shape[0], batch_size), desc="Predicting probabilities"):
        stop = min(start + batch_size, X.shape[0])
        chunks.append(model.predict_proba(X[start:stop]))
    return np.vstack(chunks)


def get_positive_scores(model, proba, positive_label=1):
    """Return the probability column for the positive class."""
    class_list = list(model.classes_)
    if positive_label not in class_list:
        raise ValueError(f"Positive label {positive_label} not found in model classes: {class_list}")
    pos_col = class_list.index(positive_label)
    return proba[:, pos_col]


def labels_from_threshold(scores, threshold, positive_label=1, negative_label=0, dtype=None):
    """Convert positive-class scores to class labels using a probability threshold."""
    pred = np.where(scores >= threshold, positive_label, negative_label)
    if dtype is not None:
        pred = pred.astype(dtype)
    return pred


def binary_skill_scores(y_true, y_pred, positive_label=1):
    """Compute common flare-forecasting binary skill scores."""
    y_true_pos = np.asarray(y_true) == positive_label
    y_pred_pos = np.asarray(y_pred) == positive_label

    tn, fp, fn, tp = confusion_matrix(
        y_true_pos,
        y_pred_pos,
        labels=[False, True],
    ).ravel()

    pod = tp / (tp + fn) if (tp + fn) else np.nan          # Probability of detection / recall
    far = fp / (tp + fp) if (tp + fp) else np.nan          # False alarm ratio
    fpr = fp / (fp + tn) if (fp + tn) else np.nan          # False positive rate
    tss = pod - fpr if np.isfinite(pod) and np.isfinite(fpr) else np.nan

    hss_denom = ((tp + fn) * (fn + tn)) + ((tp + fp) * (fp + tn))
    hss = (2 * ((tp * tn) - (fp * fn)) / hss_denom) if hss_denom else np.nan

    return {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "POD_recall": pod,
        "FAR": far,
        "FPR": fpr,
        "TSS": tss,
        "HSS": hss,
    }


def make_metrics_dict(y_true, y_pred, positive_scores=None, positive_label=1, prefix=""):
    """Return standard metrics plus flare skill scores."""
    out = {
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}precision_positive": precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        f"{prefix}recall_positive": recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        f"{prefix}f1_positive": f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
    }

    for key, value in binary_skill_scores(y_true, y_pred, positive_label=positive_label).items():
        out[f"{prefix}{key}"] = value

    if positive_scores is not None:
        y_true_binary = np.asarray(y_true) == positive_label
        out[f"{prefix}roc_auc"] = roc_auc_score(y_true_binary, positive_scores)
        out[f"{prefix}average_precision_pr_auc"] = average_precision_score(y_true_binary, positive_scores)

    return out


## 11. Tune the probability threshold on validation data

This cell finds the validation threshold that maximizes **TSS**.

It also compares that threshold against the normal `0.50` probability threshold.


In [ ]:
start_time = time.time()
val_proba = predict_proba_in_batches(quant, X_val, batch_size=PREDICT_BATCH_SIZE)
val_proba_seconds = time.time() - start_time
val_scores = get_positive_scores(quant, val_proba, positive_label=positive_label)

fpr, tpr, thresholds = roc_curve(y_val, val_scores, pos_label=positive_label)
tss_scores = tpr - fpr

valid_threshold_mask = np.isfinite(thresholds)
valid_indices = np.where(valid_threshold_mask)[0]
best_full_idx = valid_indices[np.argmax(tss_scores[valid_threshold_mask])]

best_threshold = float(thresholds[best_full_idx])
best_validation_tss = float(tss_scores[best_full_idx])
best_validation_recall = float(tpr[best_full_idx])
best_validation_fpr = float(fpr[best_full_idx])

print(f"Validation probability prediction time: {val_proba_seconds / 60:.2f} minutes ({val_proba_seconds:.1f} seconds)")
print(f"Best validation threshold for TSS: {best_threshold:.6f}")
print(f"Validation TSS at best threshold: {best_validation_tss:.4f}")
print(f"Validation recall/POD at best threshold: {best_validation_recall:.4f}")
print(f"Validation FPR at best threshold: {best_validation_fpr:.4f}")

# Build a compact threshold table. Sorting by TSS makes the top candidate thresholds easy to inspect.
threshold_results_df = pd.DataFrame({
    "threshold": thresholds,
    "validation_recall_POD": tpr,
    "validation_FPR": fpr,
    "validation_TSS": tss_scores,
})
threshold_results_df = threshold_results_df[np.isfinite(threshold_results_df["threshold"])].copy()
threshold_results_df = threshold_results_df.sort_values("validation_TSS", ascending=False).reset_index(drop=True)

print("Top 10 validation thresholds by TSS:")
display(threshold_results_df.head(10))

# Compare validation metrics at 0.50 versus the optimized threshold.
y_val_pred_default = labels_from_threshold(
    val_scores,
    DEFAULT_PROBA_THRESHOLD,
    positive_label=positive_label,
    negative_label=negative_label,
    dtype=y_val.dtype,
)
y_val_pred_tss = labels_from_threshold(
    val_scores,
    best_threshold,
    positive_label=positive_label,
    negative_label=negative_label,
    dtype=y_val.dtype,
)

validation_default_metrics = make_metrics_dict(
    y_val,
    y_val_pred_default,
    positive_scores=val_scores,
    positive_label=positive_label,
    prefix="validation_default_threshold_",
)
validation_tss_metrics = make_metrics_dict(
    y_val,
    y_val_pred_tss,
    positive_scores=val_scores,
    positive_label=positive_label,
    prefix="validation_tss_threshold_",
)

validation_comparison_df = pd.DataFrame([
    {"setting": f"default_threshold_{DEFAULT_PROBA_THRESHOLD}", **validation_default_metrics},
    {"setting": f"tss_optimized_threshold_{best_threshold:.6f}", **validation_tss_metrics},
]).T

display(validation_comparison_df)


## 12. Optional: refit on all training data after threshold selection

By default, this is skipped. Keeping it skipped is the cleanest first experiment because the probability scores on validation and test come from the same fitted model.

If `REFIT_ON_FULL_TRAIN_AFTER_THRESHOLD = True`, the threshold is still chosen only from validation data, but the model is refit on all sampled training data before test prediction.


In [ ]:
refit_seconds = 0.0

if REFIT_ON_FULL_TRAIN_AFTER_THRESHOLD:
    print("Refitting QUANT on all sampled training data using the chosen validation threshold...")
    quant = QUANTClassifier(
        estimator=ExtraTreesClassifier(
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_SEED,
        ),
        random_state=RANDOM_SEED,
    )

    start_time = time.time()
    quant.fit(X_train_fit, y_train_fit)
    refit_seconds = time.time() - start_time
    print(f"Refit time: {refit_seconds / 60:.2f} minutes ({refit_seconds:.1f} seconds)")
else:
    print("REFIT_ON_FULL_TRAIN_AFTER_THRESHOLD is False. Using the validation-tuned model for test evaluation.")


## 13. Predict on the test set with the TSS-optimized threshold

This cell computes test probabilities, then creates two test predictions:

- `y_pred_default_threshold`: threshold = 0.50
- `y_pred_tss_threshold`: threshold = validation-optimized threshold

Final reported/saved predictions use the TSS-optimized threshold.


In [ ]:
start_time = time.time()
y_proba = predict_proba_in_batches(quant, X_test_eval, batch_size=PREDICT_BATCH_SIZE)
predict_seconds = time.time() - start_time
positive_scores = get_positive_scores(quant, y_proba, positive_label=positive_label)

# Two sets of predictions for comparison.
y_pred_default_threshold = labels_from_threshold(
    positive_scores,
    DEFAULT_PROBA_THRESHOLD,
    positive_label=positive_label,
    negative_label=negative_label,
    dtype=y_test_eval.dtype,
)
y_pred_tss_threshold = labels_from_threshold(
    positive_scores,
    best_threshold,
    positive_label=positive_label,
    negative_label=negative_label,
    dtype=y_test_eval.dtype,
)

# Main prediction output for this notebook.
y_pred = y_pred_tss_threshold
threshold_used = best_threshold

print(f"Test probability prediction time: {predict_seconds / 60:.2f} minutes ({predict_seconds:.1f} seconds)")
print("Predicted probabilities shape:", y_proba.shape)
print("Class order:", quant.classes_)
print(f"Default threshold: {DEFAULT_PROBA_THRESHOLD}")
print(f"TSS-optimized threshold used on test: {threshold_used:.6f}")
print("Predictions shape:", y_pred.shape)


## 14. Evaluate default-threshold vs TSS-optimized test performance

For this imbalanced solar flare task, prioritize **TSS, HSS, POD/recall, FPR, FAR, ROC-AUC, and PR-AUC** over raw accuracy.

Threshold tuning usually increases recall and TSS, but it can also increase false positives and lower precision.


In [ ]:
print("Using positive label:", positive_label)

base_info = {
    "fit_seconds": fit_seconds,
    "val_proba_seconds": val_proba_seconds,
    "refit_seconds": refit_seconds,
    "predict_seconds": predict_seconds,
    "n_model_train_cases": int(len(y_model_train)),
    "n_validation_cases": int(len(y_val)),
    "n_train_cases_total_available_after_sampling_and_filtering": int(len(y_train_fit)),
    "n_test_cases": int(len(y_test_eval)),
    "validation_size": VALIDATION_SIZE,
    "default_threshold": DEFAULT_PROBA_THRESHOLD,
    "tss_optimized_threshold": best_threshold,
    "validation_tss_at_optimized_threshold": best_validation_tss,
    "validation_recall_at_optimized_threshold": best_validation_recall,
    "validation_fpr_at_optimized_threshold": best_validation_fpr,
    "refit_on_full_train_after_threshold": REFIT_ON_FULL_TRAIN_AFTER_THRESHOLD,
    "features_used": FEATURES_TO_USE,
    "model": "aeon.classification.interval_based.QUANTClassifier",
    "model_params": quant.get_params(),
}

default_test_metrics = {
    **base_info,
    "threshold_setting": f"default_{DEFAULT_PROBA_THRESHOLD}",
    "threshold_used": DEFAULT_PROBA_THRESHOLD,
    **make_metrics_dict(
        y_test_eval,
        y_pred_default_threshold,
        positive_scores=positive_scores,
        positive_label=positive_label,
    ),
}

optimized_test_metrics = {
    **base_info,
    "threshold_setting": "validation_tss_optimized",
    "threshold_used": best_threshold,
    **make_metrics_dict(
        y_test_eval,
        y_pred_tss_threshold,
        positive_scores=positive_scores,
        positive_label=positive_label,
    ),
}

# Main metrics object saved by the notebook.
metrics = optimized_test_metrics

comparison_df = pd.DataFrame([default_test_metrics, optimized_test_metrics])
comparison_df = comparison_df.set_index("threshold_setting")

# Display a compact view first.
compact_cols = [
    "threshold_used",
    "accuracy",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
    "POD_recall",
    "FAR",
    "FPR",
    "TSS",
    "HSS",
    "roc_auc",
    "average_precision_pr_auc",
    "TP",
    "TN",
    "FP",
    "FN",
]
compact_cols = [c for c in compact_cols if c in comparison_df.columns]
display(comparison_df[compact_cols])

print("Classification report — default 0.50 threshold:")
print(classification_report(y_test_eval, y_pred_default_threshold, zero_division=0))

print("Classification report — TSS-optimized threshold:")
print(classification_report(y_test_eval, y_pred_tss_threshold, zero_division=0))

cm_labels = np.unique(np.concatenate([y_test_eval, y_pred_default_threshold, y_pred_tss_threshold]))

cm_default = confusion_matrix(y_test_eval, y_pred_default_threshold, labels=cm_labels)
cm_default_df = pd.DataFrame(
    cm_default,
    index=[f"true_{label}" for label in cm_labels],
    columns=[f"pred_{label}" for label in cm_labels],
)
print("Confusion matrix — default 0.50 threshold")
display(cm_default_df)

cm_tss = confusion_matrix(y_test_eval, y_pred_tss_threshold, labels=cm_labels)
cm_tss_df = pd.DataFrame(
    cm_tss,
    index=[f"true_{label}" for label in cm_labels],
    columns=[f"pred_{label}" for label in cm_labels],
)
print("Confusion matrix — TSS-optimized threshold")
display(cm_tss_df)


## 15. Save predictions, metrics, threshold table, and fitted model

The model file can be large. Set `SAVE_MODEL = False` if you only want metrics and predictions.


In [ ]:
SAVE_MODEL = True

predictions_df = pd.DataFrame({
    "y_true": y_test_eval,
    "positive_probability": positive_scores,
    "y_pred_default_threshold": y_pred_default_threshold,
    "y_pred_tss_threshold": y_pred_tss_threshold,
    "threshold_used_for_tss_prediction": threshold_used,
})

print("Adding class probabilities to dataframe...")
for i, cls in enumerate(tqdm(quant.classes_, desc="Classes")):
    predictions_df[f"proba_class_{cls}"] = y_proba[:, i]

predictions_path = OUTPUT_DIR / "quant_tss_optimized_5_features_predictions.csv"
metrics_path = OUTPUT_DIR / "quant_tss_optimized_5_features_metrics.json"
comparison_metrics_path = OUTPUT_DIR / "quant_tss_optimized_5_features_metric_comparison.csv"
threshold_table_path = OUTPUT_DIR / "quant_tss_validation_threshold_table.csv"
model_path = OUTPUT_DIR / "quant_tss_optimized_5_features_model.joblib"

print("Writing predictions CSV...")
predictions_df.to_csv(predictions_path, index=False)
comparison_df.to_csv(comparison_metrics_path)
threshold_results_df.to_csv(threshold_table_path, index=False)

# Convert numpy/scikit objects to JSON-safe objects.
def make_json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, tuple):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    # Fallback for estimator objects inside model_params.
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        return str(obj)

with open(metrics_path, "w") as f:
    json.dump(make_json_safe(metrics), f, indent=2)

print("Saved predictions to:        ", predictions_path)
print("Saved optimized metrics to:  ", metrics_path)
print("Saved comparison metrics to: ", comparison_metrics_path)
print("Saved threshold table to:    ", threshold_table_path)

if SAVE_MODEL:
    print("Serializing model (this may take a moment)...")
    joblib.dump(quant, model_path)
    print("Saved fitted model to:", model_path)
else:
    print("SAVE_MODEL is False, so the fitted model was not saved.")
